# AlphaTrain — RunPod Training

Upload `alphatrain_colab_td.tar.gz` to the RunPod workspace, then run all cells.

Checkpoints saved locally to `/workspace/checkpoints/`. Download them manually when done.

In [ ]:
import os
os.chdir('/workspace')
!tar xzf alphatrain_colab_td.tar.gz
!pip install -q numpy numba pytest scipy

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Data: {os.path.getsize("/workspace/alphatrain/data/alphatrain_pairwise.pt") / 1e6:.0f} MB')
!python -m pytest alphatrain/tests/ -v --tb=short

In [ ]:
# Pure ranking: scalar value head, warm start from Pillar 2a
# Checkpoints saved locally (no Drive) — download when done
!python -m alphatrain.train \
    --tensor-file alphatrain/data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --value-bins 1 \
    --resume alphatrain/data/alphatrain_2a_best.pt --warm-start \
    --epochs 20 --batch-size 4096 --lr 3e-4 --warmup-epochs 2 \
    --rank-weight 1.0 \
    --save-dir /workspace/checkpoints/alphatrain_td

In [ ]:
# Evaluate
!python -m alphatrain.evaluate --player policy \
    --model /workspace/checkpoints/alphatrain_td/best.pt \
    --games 20 --seed 42

!python -m alphatrain.scripts.debug_value_head \
    --model /workspace/checkpoints/alphatrain_td/best.pt --seed 42

In [ ]:
# Download checkpoints: best.pt and latest.pt
!ls -lh /workspace/checkpoints/alphatrain_td/